### Simulating character interactions in Warhammer Kill Team

#### Brief project description
The goal of this project is to develop a tactical simulator for the skirmish game Warhammer Kill Team. The application will provide a statistical breakdown of combat encounters between two specific operatives, accounting for their unique stats, weapon profiles, and battlefield conditions (such as cover and special rules).

#### Project Workflow

- **Data Architecture:** Designing and populating a database with a complete dataset of Kill Team operatives and their subjective statistics, weapons and abilities.
- **Input Prototyping:** Designing a user-friendly interface to select combatants and define situational modifiers (e.g., In Cover, Injured status, near Objective, distance etc.).

- **Simulation Engine:** Developing a core algorithm that handles the combat sequence:
    - Deciding how to spend AP: decision tree. Each operative has around 3 APs.
    - Shooting: Attack/Defense dice pool resolution.
    - Fighting: A decision-tree-based logic for melee combat (Strike vs. Parry).
    - Special Rules: Implementation of rerolls (e.g., Ceaseless, Relentless) and critical effects.

- **Data Analysis & Output:** Transforming raw simulation data into meaningful statistical values and visual graphs to show the probability of various outcomes.
- **GUI Implementation or user-friendly CLI**


#### What is Warhammer Kill Team?
Warhammer Kill Team is a tactical, small-scale skirmish game set in the **grimdark universe of Warhammer 40,000**. Unlike the large-scale army battles of the main game, Kill Team focuses on elite squads of roughly 5 to 14 individual operatives. Each model acts independently with its own specific characteristics, specialist roles, and action points. The game is played on smaller boards with dense terrain, emphasizing intense, **fast-paced combat** and strategic depth.

#### How does a round look from perspective of character?
In Kill Team, a character's turn is driven by their Action Point Limit (APL).
1. **Activation:** You select a model and choose between an Engage order (to attack) or a Conceal order (to hide).
2. **Spending AP:** You spend your Action Points on maneuvers like moving, dashing, or interacting with mission objectives.
3. **Combat:** You can perform a Shoot action against distant enemies or a Fight action if you are in base-to-base contact.
4. **Resolution:** Attacks are resolved by rolling a pool of dice, with both the attacker and defender rolling to determine damage.
5. **Completion:** Once all points are spent, the model is marked as activated, and the turn passes to the opponent.

#### Example duel round: Space Marine Sergeant vs Plague Marine Champion (SMS vs PMC), distance between them 6", no cover
**SMS starts, has 3AP, special ability - can perform shoot action twice, range 6"**
1. SMS shoots for 1AP
    - chooses classic bolter weapon (ATK 4, HIT 2+, DMG 3/4, piercing crits 1)
    - rolls 4 dice, results: 1,1,6,2 (resolve: 1 a 1 fail; 2 valid normal hit for 3 wounds; 6 critical hit for 4 wounds and activates piercing crits 1)
    - PMC throws 3 defense dice ( needs to hit 3+), results: 4,3,1 (resolve: 1 fail; 3 a 4 valid normal saves, **piercing crits** discards one valid save --> only one counts)
    - SMS attacks with 6 and 2, PMC saves only with 4 --> PMC gets 4 wounds from the critical, now has 11 wounds
2. SMS shoots again for 1AP (because of special ability)
    - similar resolving of throws
3. SMS has 1 AP left, but cannot perform any action.

**PMC is activated, has 3AP, range 8", reposition range 5"**
1. PMC repositions himself to control range (1") of SMS, costs 1AP
2. fights with SMS, costs 1AP (this turn is simplified in this guide)
    - PMC rolls 5 dice for his, needs to hit 3+ (from stats)
    - SMS also rolls his close-combat weapon, but only 4 dice, needs to hit 3+
    - now they fight with them, f. e. PMC has 3 valid dice and SMS has 2
    - PMC starts, he is the attacking one, he has to attack with his dice
    - SMS now has to choose: block it or let PMC hit him.
        - he blocks, PMC attacks again.
        - he lets it hit, now SMS attacks and PMC decides whether to block.
3. PMC has 1 AP left, he can fight again (special ability)

## Deep Tactical Engine Implementation

### Step 0: Memory Management & Duplicate Prevention
**Technical term:** *Zobrist Hashing & Transposition Tables*

Before running the search tree, a hash table is initialized. Each unique combination of operative positions and their remaining HP is assigned a unique 64-bit key. If the engine reaches the same game state via different move sequences, it skips redundant calculations and retrieves the evaluation from memory.

### Step 1: Goal-Oriented Space Discretization
**Technical term:** *Goal-Oriented State Space Discretization & Vector Raycasting*

Map geometry is converted into a finite set of logical points (cover edges, engagement perimeters, objective radii). This step is directly modulated by the tactical goal (KILL, SURVIVE, CAPTURE). For aggressive goals, passive locations are ignored, eliminating irrelevant states before the tree is even built.

### Step 2: One-Way State Machine & Space Pruning
**Technical term:** *One-Way State Machine & Heuristic Beam Search*

Operative activation is split into two irreversible phases. In the **positional phase**, the algorithm spends AP exclusively on moves. It then applies a heuristic equation evaluating cover and LOS, discarding the bottom 80% of sub-optimal nodes. Only the "Beam" (Top 200) passes to the action phase.

### Step 3: Terminal Action Expansion & Reliable Evaluation
**Technical term:** *Action Expansion & Trimmed Mean Evaluator (P25)*

In terminal nodes, interaction outcomes (Shoot, Fight) are calculated. Instead of naive Expected Value, the engine uses probability matrices and scales results (e.g., P25 percentile). This ensures moves are quantified by their reliable core rather than statistical outliers.

### Step 4: Isolated Adversarial Threat Analysis
**Technical term:** *Adversarial Lookahead (1-ply Minimax)*

The engine simulates the opponent's reaction from each end state. To avoid "Hive Fallacy" (paralysis from summing all possible enemy attacks), threats are evaluated in isolation. The algorithm identifies the single most dangerous enemy profile for the current goal and subtracts that threat from the node's score.

### Step 5: Time-Bounded Search Iteration
**Technical term:** *Time-Bounded Iterative Deepening*

The algorithm doesn't wait for a fixed depth. It immediately selects top branches and repeats the process for 2-ply, 3-ply, etc., incorporating subsequent activations. The loop runs until the 20-second hard timer terminates the process.

### Step 6: Multi-Criteria Decision Analysis
**Technical term:** *MCDA with Linear Utility Function*

Once time expires, the deepest evaluated nodes are processed. Hard data (confirmed damage, VP gain, quantified risk) is multiplied by target-specific weights to return the Top 5 most stable and reliable action sequences to the GUI.

## Version1: How to simulate the game? (not used in the final project)

### What  we get from the user?
- target of his action: KILL, DEAL DMG, CAPTURE, SURVIVE/FLEE
- operative + map layout + remaining wounds + active operative

### Output of the simulation
- top 5 actions based on his target
- the top one showing on the map
- he can download a file with recommended steps / shown in gui

### Action Evaluation
- value every step, how good it is based on the target action
    - score = (damage * wd) + (kills * wk) + (objective * wo) + (position * wp) - (risk * wr)
    - calculate it based on weight by target

#### weights by target:

*KILL:* damage = 0.8, kills = 1.0, objective = 0.2, position = 0.4, risk = 0.3

*DAMAGE:* damage = 1.0, kills = 0.7, objective = 0.2, position = 0.3, risk = 0.2

*CAPTURE:* damage = 0.5, kills = 0.2, objective = 1.0, position = 0.5, risk = 0.4

*SURVIVE:* damage = 0.4, kills - 0.1, objective = 0.3, position = 0.7, risk = 1.0

#### how to calculate each part?

*Damage:* Expected = N_dice * P(hit) * ( 1-P(save) ) * ( dmg / hit )

*Kills:* Expected >= HP, kill = 1, else 0. it is a fixed bonus

*Objective* Score = max(0, 100-(distance^2) ), square => closer = better

*Position* Score = Sobj + Slos*0.9 + Stac*1.2 + Scov*1.5

- Sobj = 100 - ( dist(active_op, obj) * 10)
- Slos = SUM(visible_enemy * 20) + SUM(covered_enemy * 10)
- Stac = Obj -> Me -> Enemy => +30, Me->Friend to 2" => + 30, Weak Friend-> Me -> Enemy => + 25
- Scov => SUM(enemy * 15)
    - standing between objective and enemy ++
    - standing between enemy and friendly unit ++
    - close range to friendly unit ++
    - close to objective ++
    - in cover ++
    - visible targets w/o cover ++
    - visible targets with cover +


*Risk* Score = SUM (base * cover * threat * wounds) ^ 1.4
- base = 1
- cover = 0.65 | 1 | 0  (not visible | conceal => 0)
- wounds = >75% 0.75, >50% 1, <50% 1.25, <25% 1,5
- threat table:
    - charge range of enemy with basic melee = 2   (cover = 1)
    - charge range of enemy with strong melee = 3   (cover = 1)
    - shoot range of enemy with basic ranged = 1.5
    - shoot range of enemy with better ranged = 2.5

#### Normalized

*Damage* = min (100, (EV/12) * 100 )

*Kills* = 100 * Kill_amount (<= 1)

*Objective* = max(0, 100-(distance^2) )

*Position* = min (100, Position * 0.5 ))

*Risk* = min (150, (SUM (base * cover * threat * wounds) ^ 1.4)


#### divide the map into grid of small points
- for every point calculate:
    -  threat index
    - strategic value index (see an enemy, cover, dist to points, etc.)
- than calculate total strategic value of this point



### How to resolve the actions?
##### how many actions are possible?
    - every operative has 3 AP:
      1. move | shoot | capture | fall back
      2. move -> shoot | move (move + 2" = charge) -> fight | move -> capture
      3. move -> dash -> shoot | capture
      4. move -> capture -> shoot
      5. fall back + shoot | capture
      6. move -> 2x shoot | 2x fight ( ability)

#### Option A: ranged/capture
1. generate possible points to move to in radius of Move + Dash
    - into struct {x: 0, y: 0, remaining AP: 1-3, actions: list, shoot phase: list of objects, captured: bool, indexes ..., }
2. filter out collisions with obstacles  (also during the move) or 1" within enemy / friendly operative
3. calculate risk + position index
4. simulate shooting phase and save into s-phase = {target, dmg done, weapon, dice prob, save prob, remaining wounds} save top 5 into struct, calc idx
5. simulate capture if possible, calc idx

#### Option B: fight
1. generate possible points to move to in radius of Charge. need to Charge an enemy operative ( impact point <= 1")
    - into struct {x: 0, y: 0, remaining AP: 1-3, actions: list, fight phase: list of objects, captured: bool, indexes ..., }
2. filter out collisions with obstacles / operatives (also during the move)
3. calculate risk + position index
4. simulate fight and save into f-phase = {target, dmg done, dmg received, weapon, dice prob, save prob, remaining wounds} save top 5 into struct, calc idx
    - if receives big dmg add it to risk index
        -
#### Resolution
choose top 5-10 variants based on valuation and show them to the user

### Shooting Phase Simulation
with every possible weapon
1. Resolve pre-throw abilities of weapons & operative, which affect the dice
2. Calculate probability of hits ( dont forget the post roll effects)
3. Calculate probability of save (post positive/negative effects)
4. calculate total probability of hit and avg total dmg
5. return s-phase object + received and dealt wounds & kills

### Fighting Phase Simulation
with every possible weapon
1. Resolve pre-throw abilities of weapons & operative, which affect the dice
2. roll attacker dice, resolve effects
3. roll defender dice, resolve effects
4. attacker always tries to do as much dmg as possible, if he is able to defend himself also, he does it
5. defender blocks, if he knows he is going to die, try to do as much damage as possible
6. calculate the probabilities and resolve the phase
7. return f-phase object + received and dealt dmg + kills